# ETAPA 1: Análisis Exploratorio de Multas de Tráfico

## Librerías

In [20]:
import io
import pandas as pd
import requests
from pandas import DataFrame
from pandas.io.parsers import TextFileReader

### Descarga de datos

Cargamos los datos de diciembre de 2024 desde el portal de datos abiertos del Ayuntamiento de Madrid.

In [2]:
url = "https://datos.madrid.es/egob/catalogo/210104-395-multas-circulacion-detalle.csv"

response = requests.get(url)
if response.status_code == 200:
    print(f"URL final tras redirección: {response.url}")
else:
    print(f"Error HTTP: {response.status_code}")

URL final tras redirección: https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv


In [3]:
content = io.StringIO(response.text)
multas: TextFileReader | DataFrame = pd.read_csv(content, sep=';', encoding='latin1')
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA. ...,,,,
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",,,,
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACI�N DE UN...,,,,
3,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA. ...,,,,
4,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",,,,


### Explorar la estructura de los datos

In [4]:
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column                                                                                                                                                  Non-Null Count   Dtype  
---  ------                                                                                                                                                  --------------   -----  
 0   CALIFICACION                                                                                                                                            249801 non-null  str    
 1   LUGAR                                                                                                                                                   249801 non-null  str    
 2   MES                                                                                                                                                     249801 non-null

**Comentarios del resultado de `info()`:**

- El dataframe tiene **14 columnas** y el número de filas se muestra en la cabecera.
- Las columnas `MES` y `ANIO` son de tipo `int64`, lo cual es correcto para valores numéricos de fecha.
- La columna `HORA` es de tipo `float64` (formato decimal, ej. `20.23` para las 20:23 h).
- La columna `IMP_BOL` es de tipo `float64`, representando el importe en euros.
- Las columnas `VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X` y `COORDENADA-Y` aparecen como `object` (texto),
  porque contienen celdas vacías que impiden la inferencia automática de tipo numérico.
- Las columnas de texto como `CALIFICACION`, `DENUNCIANTE`, `HECHO-BOL` presentan muchos espacios en blanco
  al inicio y al final, lo que puede dificultar agrupaciones y comparaciones.
- Varios campos tienen valores `NaN` (nulos), especialmente los relacionados con velocidad y coordenadas,
  que solo se rellenan en infracciones de velocidad.

### Exploración de las columnas

#### a) Columna `CALIFICACION`

In [5]:
# Eliminar espacios en blanco
multas['CALIFICACION'] = multas['CALIFICACION'].str.strip()

print(f"Valores distintos en CALIFICACION: {multas['CALIFICACION'].nunique()}")
print(multas['CALIFICACION'].unique())

Valores distintos en CALIFICACION: 3
<StringArray>
['LEVE', 'GRAVE', 'MUY GRAVE']
Length: 3, dtype: str


#### b) Columnas `DESCUENTO`, `HECHO-BOL` y `DENUNCIANTE`

In [6]:
for col in ['DESCUENTO', 'HECHO-BOL', 'DENUNCIANTE']:
    print(f"\n--- {col} ---")
    multas[col] = multas[col].str.strip()
    print(f"Valores distintos: {multas[col].nunique()}")


--- DESCUENTO ---
Valores distintos: 1

--- HECHO-BOL ---
Valores distintos: 345

--- DENUNCIANTE ---
Valores distintos: 6


#### c) Normalización de nombres de columnas

In [7]:
print("Listado de columnas del dataframe:")
print(list(multas.columns))

Listado de columnas del dataframe:
['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO', ' PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA ', 'COORDENADA-X', 'COORDENADA-Y                                                                                                                                          ']


In [8]:
# La columna de velocidad de circulación tiene un espacio al final
col_vel = [c for c in multas.columns if 'VEL_CIRCULA' in c][0]
print(f"Nombre exacto de VEL_CIRCULA: {repr(col_vel)}")

# La coordenada Y tiene varios espacios al final
col_coord_y = [c for c in multas.columns if 'COORDENADA-Y' in c][0]
print(f"Nombre exacto de COORDENADA-Y: {repr(col_coord_y)}")

Nombre exacto de VEL_CIRCULA: 'VEL_CIRCULA '
Nombre exacto de COORDENADA-Y: 'COORDENADA-Y                                                                                                                                          '


In [9]:
# Mostrar la serie de datos de la coordenada Y
print("Serie COORDENADA-Y (primeros valores):")
print(multas[col_coord_y].head(10))
print(f"\nValores distintos: {multas[col_coord_y].nunique()}")

Serie COORDENADA-Y (primeros valores):
0               
1               
2               
3               
4               
5               
6               
7               
8               
9               
Name: COORDENADA-Y                                                                                                                                          , dtype: str

Valores distintos: 2500


In [10]:
# Normalizar todos los nombres de columnas eliminando espacios al final
multas.rename(columns=str.strip, inplace=True)

print("Columnas normalizadas:")
print(list(multas.columns))

Columnas normalizadas:
['CALIFICACION', 'LUGAR', 'MES', 'ANIO', 'HORA', 'IMP_BOL', 'DESCUENTO', 'PUNTOS', 'DENUNCIANTE', 'HECHO-BOL', 'VEL_LIMITE', 'VEL_CIRCULA', 'COORDENADA-X', 'COORDENADA-Y']


#### d) Columnas de velocidad: conversión a numérico

In [11]:
print(f"Tipo de VEL_LIMITE antes: {multas['VEL_LIMITE'].dtype}")
print(f"Tipo de VEL_CIRCULA antes: {multas['VEL_CIRCULA'].dtype}")
print(f"\nValores distintos en VEL_LIMITE: {multas['VEL_LIMITE'].nunique()}")
print(multas['VEL_LIMITE'].value_counts().head())
print(f"\nValores distintos en VEL_CIRCULA: {multas['VEL_CIRCULA'].nunique()}")
print(multas['VEL_CIRCULA'].value_counts().head())

Tipo de VEL_LIMITE antes: str
Tipo de VEL_CIRCULA antes: str

Valores distintos en VEL_LIMITE: 7
VEL_LIMITE
      227619
70     10235
50      5153
90      4261
60      1758
Name: count, dtype: int64

Valores distintos en VEL_CIRCULA: 101
VEL_CIRCULA
      227619
74      1903
75      1635
76      1359
77      1168
Name: count, dtype: int64


In [12]:
# Convertir a numérico (los valores vacíos se convertirán en NaN)
multas['VEL_LIMITE']  = pd.to_numeric(multas['VEL_LIMITE'],  errors='coerce')
multas['VEL_CIRCULA'] = pd.to_numeric(multas['VEL_CIRCULA'], errors='coerce')

print(f"Tipo de VEL_LIMITE después: {multas['VEL_LIMITE'].dtype}")
print(f"Tipo de VEL_CIRCULA después: {multas['VEL_CIRCULA'].dtype}")

Tipo de VEL_LIMITE después: float64
Tipo de VEL_CIRCULA después: float64


#### e) Columnas de geolocalización: conversión a numérico

In [13]:
print(f"Tipo de COORDENADA-X antes: {multas['COORDENADA-X'].dtype}")
print(f"Tipo de COORDENADA-Y antes: {multas['COORDENADA-Y'].dtype}")

Tipo de COORDENADA-X antes: str
Tipo de COORDENADA-Y antes: str


In [14]:
multas['COORDENADA-X'] = pd.to_numeric(multas['COORDENADA-X'], errors='coerce')
multas['COORDENADA-Y'] = pd.to_numeric(multas['COORDENADA-Y'], errors='coerce')

print(f"Tipo de COORDENADA-X después: {multas['COORDENADA-X'].dtype}")
print(f"Tipo de COORDENADA-Y después: {multas['COORDENADA-Y'].dtype}")

Tipo de COORDENADA-X después: float64
Tipo de COORDENADA-Y después: float64


In [15]:
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   CALIFICACION  249801 non-null  str    
 1   LUGAR         249801 non-null  str    
 2   MES           249801 non-null  int64  
 3   ANIO          249801 non-null  int64  
 4   HORA          249801 non-null  float64
 5   IMP_BOL       249801 non-null  float64
 6   DESCUENTO     249801 non-null  str    
 7   PUNTOS        249801 non-null  int64  
 8   DENUNCIANTE   249801 non-null  str    
 9   HECHO-BOL     249801 non-null  str    
 10  VEL_LIMITE    22182 non-null   float64
 11  VEL_CIRCULA   22182 non-null   float64
 12  COORDENADA-X  123882 non-null  float64
 13  COORDENADA-Y  123882 non-null  float64
dtypes: float64(6), int64(3), str(5)
memory usage: 26.7 MB


### Creación de columna `fecha` y uso como índice

#### a) Crear columna `fecha` de tipo datetime

In [16]:
# La HORA está en formato HH.MM (NO decimal):
#   20.23 → hora=20, minuto=23  (los decimales son minutos, no fracción de hora)
#    9.05 → hora=9,  minuto=5

def hora_a_hm(hora):
    try:
        h = float(hora)
        hora_int   = int(h)
        minuto_int = round((h - hora_int) * 100)  # .23 → 23 minutos
        if not (0 <= minuto_int <= 59):
            minuto_int = 0
        return hora_int, minuto_int
    except (ValueError, TypeError):
        return 0, 0

hm            = multas['HORA'].apply(hora_a_hm)
hora_entera   = hm.apply(lambda x: x[0])
minuto_entero = hm.apply(lambda x: x[1])

multas['fecha'] = pd.to_datetime(
    {
        'year':   multas['ANIO'],
        'month':  multas['MES'],
        'day':    1,
        'hour':   hora_entera,
        'minute': minuto_entero,
    },
    errors='coerce'
)

print(f"Tipo de la columna fecha: {multas['fecha'].dtype}")
print(multas['fecha'].head(1))

Tipo de la columna fecha: datetime64[us]
0   2024-12-01 20:23:00
Name: fecha, dtype: datetime64[us]


#### b) Establecer `fecha` como índice

In [17]:
multas.set_index('fecha', inplace=True)
print(f"Índice del dataframe: {multas.index.name}")
print(f"Tipo del índice: {multas.index.dtype}")
multas.head()

Índice del dataframe: fecha
Tipo del índice: datetime64[us]


,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
fecha,,,,,,,,,,,,,,
2024-12-01 20:23:00,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA.,NaN,NaN,NaN,NaN
2024-12-01 20:27:00,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",NaN,NaN,NaN,NaN
2024-12-01 20:45:00,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACI�N DE UN...,NaN,NaN,NaN,NaN
2024-12-01 16:30:00,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACI�N NO V�LIDA.,NaN,NaN,NaN,NaN
2024-12-01 16:50:00,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACI�...",NaN,NaN,NaN,NaN


#### c) Eliminar columna `fecha` redundante (si quedara)

In [18]:
if 'fecha' in multas.columns:
    multas.drop(columns=['fecha'], inplace=True)
    print("Columna 'fecha' eliminada del dataframe.")
else:
    print("La columna 'fecha' ya es el índice; no hay duplicado en columnas.")

print(f"\nEstructura final del dataframe:")
multas.info()

La columna 'fecha' ya es el índice; no hay duplicado en columnas.

Estructura final del dataframe:
<class 'pandas.DataFrame'>
DatetimeIndex: 249801 entries, 2024-12-01 20:23:00 to 2024-12-01 12:51:00
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   CALIFICACION  249801 non-null  str    
 1   LUGAR         249801 non-null  str    
 2   MES           249801 non-null  int64  
 3   ANIO          249801 non-null  int64  
 4   HORA          249801 non-null  float64
 5   IMP_BOL       249801 non-null  float64
 6   DESCUENTO     249801 non-null  str    
 7   PUNTOS        249801 non-null  int64  
 8   DENUNCIANTE   249801 non-null  str    
 9   HECHO-BOL     249801 non-null  str    
 10  VEL_LIMITE    22182 non-null   float64
 11  VEL_CIRCULA   22182 non-null   float64
 12  COORDENADA-X  123882 non-null  float64
 13  COORDENADA-Y  123882 non-null  float64
dtypes: float64(6), int64(3), str(5)
memory usage: 28.6 MB


### Resumen del preprocesamiento

Tras la Etapa 1 el dataframe `multas` tiene las siguientes características:

- **Índice**: columna `fecha` de tipo `datetime64[ns]`.
- **Columnas de texto**: `CALIFICACION`, `DESCUENTO`, `HECHO-BOL`, `DENUNCIANTE` sin espacios en blanco.
- **Columnas numéricas**: `VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X`, `COORDENADA-Y` de tipo `float64`.
- **Nombres de columnas**: normalizados sin espacios al inicio ni al final.

In [19]:
multas.describe(include='all')

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
count,249801,249801,249801.0,249801.0,249801.000000,249801.000000,249801,249801.000000,249801,249801,22182.000000,22182.000000,123882.000000,1.238820e+05
unique,3,34647,NaN,NaN,NaN,NaN,1,NaN,6,345,NaN,NaN,NaN,NaN
top,GRAVE,CALLE ALCALA 51,NaN,NaN,NaN,NaN,SI,NaN,MEDIOS DE CAPTACION DE IMAGEN,NO RESPETAR LAS RESTRICCIONES DE CIRCULACI�N D...,NaN,NaN,NaN,NaN
freq,157605,7440,NaN,NaN,NaN,NaN,249801,NaN,119981,60650,NaN,NaN,NaN,NaN
mean,NaN,NaN,12.0,2024.0,13.873079,148.138839,NaN,0.168838,NaN,NaN,67.351456,79.485889,428445.250604,4.345018e+06
std,NaN,NaN,0.0,0.0,4.944544,73.415946,NaN,0.808905,NaN,NaN,14.225886,13.991656,73459.914494,7.440747e+05
min,NaN,NaN,12.0,2024.0,0.000000,30.000000,NaN,0.000000,NaN,NaN,30.000000,44.000000,4330.060000,4.466931e+04
25%,NaN,NaN,12.0,2024.0,10.500000,90.000000,NaN,0.000000,NaN,NaN,50.000000,72.000000,439271.020000,4.470684e+06
50%,NaN,NaN,12.0,2024.0,13.570000,200.000000,NaN,0.000000,NaN,NaN,70.000000,77.000000,440653.200000,4.473581e+06
75%,NaN,NaN,12.0,2024.0,17.520000,200.000000,NaN,0.000000,NaN,NaN,70.000000,87.000000,442243.640000,4.475503e+06
